In [46]:
import pandas as pd
%run Ratios.ipynb

oh dear
                  ticker    Quarter     y  m  q  group
ticker Quarter                                        
ADBE   2009-02-28   ADBE 2009-02-28  2009  2  0      0
AAPL   2009-03-31   AAPL 2009-03-31  2009  3  0      0
NVDA   2009-01-31   NVDA 2009-01-31  2009  1  0      0
BAM    2009-03-31    BAM 2009-03-31  2009  3  0      0
NEE    2009-03-31    NEE 2009-03-31  2009  3  0      0
...                  ...        ...   ... .. ..    ...
MTZ    2009-03-31    MTZ 2009-03-31  2009  3  0      0
NEON   2009-03-31   NEON 2009-03-31  2009  3  0      0
STCN   2009-01-31   STCN 2009-01-31  2009  1  0      0
VIAV   2009-03-31   VIAV 2009-03-31  2009  3  0      0
REFR   2009-03-31   REFR 2009-03-31  2009  3  0      0

[2389 rows x 6 columns]
                  ticker    Quarter     y  m  q  group
ticker Quarter                                        
ADBE   2009-05-31   ADBE 2009-05-31  2009  5  1      1
AAPL   2009-06-30   AAPL 2009-06-30  2009  6  1      1
NVDA   2009-04-30   NVDA 2009-04

[3418 rows x 6 columns]
                  ticker    Quarter     y  m  q  group
ticker Quarter                                        
SABR   2016-03-31   SABR 2016-03-31  2016  3  0     28
ADBE   2016-02-29   ADBE 2016-02-29  2016  2  0     28
AAPL   2016-03-31   AAPL 2016-03-31  2016  3  0     28
NVDA   2016-01-31   NVDA 2016-01-31  2016  1  0     28
AMCX   2016-03-31   AMCX 2016-03-31  2016  3  0     28
...                  ...        ...   ... .. ..    ...
NEON   2016-03-31   NEON 2016-03-31  2016  3  0     28
STCN   2016-01-31   STCN 2016-01-31  2016  1  0     28
VIAV   2016-03-31   VIAV 2016-03-31  2016  3  0     28
REFR   2016-03-31   REFR 2016-03-31  2016  3  0     28
LILAK  2016-03-31  LILAK 2016-03-31  2016  3  0     28

[3287 rows x 6 columns]
                  ticker    Quarter     y  m  q  group
ticker Quarter                                        
SABR   2016-06-30   SABR 2016-06-30  2016  6  1     29
ADBE   2016-05-31   ADBE 2016-05-31  2016  5  1     29
AAPL   2016-06-3

KeyError: "['QUBT' 'LCFY' nan 'UTME' 'OST' 'SMR'] not in index"

KeyError: "['QUBT' 'LCFY' nan 'UTME' 'OST' 'SMR'] not in index"

In [213]:
class Setup(Ratios):
    
    # getUniverse
    # Function to generate the list of tickers to use as a Universe of stocks to consider
    def getUniverse():
        tickersbs = Ratios.balanceq.index.get_level_values(0)
        tickersinc = Ratios.incq.index.get_level_values(0)
        tickerscf = Ratios.cfq.index.get_level_values(0)
        
        utilities = Ratios.sectorTickers('Utilities')
        financials = Ratios.sectorTickers('Financials')
        
        universe = set(tickersinc).intersection(set(tickersbs)).intersection(set(tickerscf))
        universe = universe - set(utilities) - set(financials)
        
        # also remove tickers for which we currently have no prices...
        prices = Ratios.prices.index.get_level_values(0)
        universe = universe.intersection(set(prices))
        
        # ...and for testing just limit to a small handful of stocks because, you know...
        tech = Ratios.industryTickers('Internet Services')
        universe = universe.intersection(set(tech))
        
        return list(universe)
    
    # writeData
    # Pass in a function to perform on all tickers - executes the function and writes the result directly to a file
    # Means we shouldn't need to keep running the expensive calculations many times (though consider if universe has changed)
    def writeData(fn):
        universe = Setup.getUniverse()
        data = fn(universe)
        data.index.name = 'Quarter'
        fnname = fn.__name__
        data.to_csv(f'MT/mf-{fnname}.csv')

    # Initialisation - this should not need to be called again, just storing in the class as a record of what's been done
    def init():
        print('Calculating/writing Earnings Yield')
        Setup.writeData(Ratios.earningsYield)
        print('Calculating/writing ROC')
        Setup.writeData(Ratios.ROC)

Setup.writeData(Ratios.ROC)


In [ ]:
# MAGIC FORMULA
#
# https://einvestingforbeginners.com/return-on-capital-daah/
#
# The only item not available from MT is in the Working Capital sum
# Liabilities are not broken down very well in MT - so can only really use total Current Liabilities in the Working Capital part
# This might not be so bad as long as it applies to all companies - might be slightly off...
#
# Net Fixed Assets: propertyPlantEquipment (have to ignore depreciation as we don't have this)
# Working Capital: Some assets - Some liabilities
# Some assets: Accounts receivable + Inventories + Other current assets
# 
# Some liabilities: Accounts Payable + Income Taxes payable + Accrued Compensation + Deferred Revenue + Other current liabilities
#

# Not sure if the results from MT are any good, but perseverse for now assuming the numbers are ok
# It seems like its also possible to be more vague with the calculations
#
# eg Following a youTube tutorial - he uses ROA instead of ROC so that's immediately a far simpler approach
#
# THE MAGIC FORMULA:
# Generate a score for the earnings yield (value) and the return on capital (quality)
# Exclude Utilities and Financials
# Exclude foreign stocks (I think I already am)
# Exclude P/E < 5 - seems that these might be weird or low data-quality just remove
# Exclude recent earnings eg within one week
# Create a value rank and a quality rank then combine (how?) ADD THE RANKS TOGETHER.
# Buy top stocks either in one lump or occasionally every 2-3 months
# Hold for at least one year - periodically balance...


In [196]:
# Magic Formula class
# Contains all the functions required to run the Magic Formula over our Universe of tickers
# Call Setup.getUniverse() for the current ticker Universe
class MF():
    
    # Read source data files for the data we need
    ey = pd.read_csv('MT/mf-earningsYield.csv', parse_dates=['Quarter']).set_index('Quarter').sort_index()
    roi = pd.read_csv('MT/mf-ROI.csv', parse_dates=['Quarter']).set_index('Quarter').sort_index()

    # rankGroup
    # For a given group of tickers, representing a single quarter, generate a series ranked in order highest to lowest
    def rankGroup(g):
        collapse = g.sum(axis=0, min_count=1)
        removenas = collapse.dropna()
        ranked = removenas.sort_values(ascending=False).reset_index().reset_index().rename(columns={'level_0':'rank', 'index':'ticker',0:'metric'})
        return ranked

    # backTest
    # Run a full test on the data using fixed MF calculations
    def backTest():
        
        # First for earnings yield
        year = ey.index.year
        month = ey.index.month
        quarter = (month-1)//3
        group = 4*(year - 2009) + quarter

        eyres = ey.groupby(group, axis=0).apply(rankGroup)
        #return eyres
    
        # Similar for ROIC
        year = roi.index.year
        month = roi.index.month
        quarter = (month-1)//3
        group = 4*(year - 2009) + quarter

        roires = roi.groupby(group, axis=0).apply(rankGroup)
        return roires

In [214]:
ey = pd.read_csv('MT/mf-earningsYield.csv', parse_dates=['Quarter']).set_index('Quarter').sort_index()
roc = pd.read_csv('MT/mf-ROC.csv', parse_dates=['Quarter']).set_index('Quarter').sort_index()

In [215]:
year = ey.index.year
month = ey.index.month
quarter = (month-1)//3
group = 4*(year - 2009) + quarter

eyres = ey.groupby(group, axis=0).apply(rankGroup)
sample = eyres.loc[6]

m,s = (sample.mean(), sample.std())
m, s
sample

,rank,ticker,metric
0,0,WHLM,46.730852
1,1,TAOP,21.285210
2,2,SOHU,12.021586
3,3,GOOGL,7.001256
4,4,HSTM,6.927263
5,5,MLKN,6.323679
6,6,LPSN,3.550554
7,7,AKAM,2.681625
8,8,BIDU,1.401965
9,9,MCHX,-2.915655


In [219]:
# Repeat above for ROI
year = roi.index.year
month = roi.index.month
quarter = (month-1)//3
group = 4*(year - 2009) + quarter

roires = roi.groupby(group, axis=0).apply(rankGroup)
roires.loc[3]

,rank,ticker,metric
0,0,TAOP,18.937151
1,1,MLKN,12.384909
2,2,HSTM,9.883204
3,3,BLIN,4.365001


In [217]:
# Repeat above for ROC
year = roc.index.year
month = roc.index.month
quarter = (month-1)//3
group = 4*(year - 2009) + quarter

rocres = roc.groupby(group, axis=0).apply(rankGroup)
rocres.loc[7]

,rank,ticker,metric
0,0,TAOP,3128.310945


In [221]:
# We now have eyres and roires calculated - add ranks together for each ticker
# Ticker must appear in both otherwise it should be dropped
ey1 = eyres.loc[3]
roi1 = roires.loc[3]
merge = ey1.merge(roi1, how='inner', on='ticker')
merge['finalRank'] = merge['rank_x'] + merge['rank_y']
final = merge.sort_values(by='finalRank')
len(final)

4